# 06 — Linear and logistic regression

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omarahmedZa/ai-ml-course/blob/main/notebooks/006-linear-and-logistic-regression.ipynb)

Regression is one idea — pick the line with the smallest squared error — applied to two kinds of target. When the target is a number you read the line's height; when the target is yes/no you pass that height through a squashing function and read a probability.

## 1. The numbers: one line through ten points

Ten houses, one feature each, and a price that is roughly three times the size plus five. `np.linalg.lstsq` returns the slope and intercept of the line with the smallest total squared vertical distance to the points, with no loop and no learning rate.

In [ ]:
import numpy as np                                        # the array library
x = np.arange(1, 11, dtype=float)                         # feature: size, in hundreds of square feet
y = 3.0 * x + 5.0 + np.array([-0.2, 0.4, -0.1, 0.3, -0.5, 0.2, 0.1, -0.3, 0.6, -0.4])   # price plus noise
A = np.column_stack([x, np.ones_like(x)])                 # design matrix: a size column, then a column of ones
coef = np.linalg.lstsq(A, y, rcond=None)[0]               # [slope, intercept] with the smallest squared error
print("A shape:", A.shape)                                # ten rows, two columns
print("row 0 of A:", A[0])                                # one row: the size, then the constant 1
print("coefficients:", np.round(coef, 3))                 # slope first, intercept second
print("first three predictions:", np.round(A[:3] @ coef, 3))   # the model is A @ coef
print("last three targets:", np.round(y[-3:], 3))         # the values it approximates

**Line by line**

- `import numpy as np` — loads NumPy.
- `x = ...` — the single feature, sizes 1 to 10.
- `y = ...` — prices from a straight rule, plus small noise.
- `A = np.column_stack(...)` — the two columns the line needs: the feature and a column of ones.
- `coef = np.linalg.lstsq(...)[0]` — least squares solution, slope then intercept.
- `print("A shape" ...)` — ten rows, two unknowns.
- `print("row 0 of A" ...)` — the ones column is how an intercept gets fitted.
- `print("coefficients" ...)` — should land near 3 and 5.
- `print("first three predictions" ...)` — `A @ coef` applies the line to every row at once.
- `print("last three targets" ...)` — compare predictions with the truth.

## 2. Step by step: the same answer without lstsq

Least squares has a closed form: build the two-by-two matrix of sums, build the vector of sums, and solve. Accumulating those sums one row at a time shows exactly what `lstsq` is doing, and it lands on the identical slope and intercept.

In [ ]:
AtA = np.zeros((2, 2))                                    # will hold the summed outer products
Aty = np.zeros(2)                                         # will hold the summed feature-times-target
for i in range(len(y)):                                   # one iteration per row
    row = A[i]                                            # this row: [size, 1]
    AtA = AtA + np.outer(row, row)                        # add this row's two-by-two outer product
    Aty = Aty + row * y[i]                                # add this row's contribution to the right side
    print(f"row {i}: AtA[0,0]={AtA[0,0]:.1f}  Aty={np.round(Aty, 1)}")   # watch the sums accumulate
w_hand = np.linalg.solve(AtA, Aty)                        # solve the two equations for slope and intercept
print("by hand:", np.round(w_hand, 3))                    # slope, intercept
print("lstsq  :", np.round(coef, 3))                      # the same numbers, reached differently

**Line by line**

- `AtA = np.zeros((2, 2))` — the 2-by-2 matrix that will collect the sums.
- `Aty = np.zeros(2)` — the 2-vector on the right-hand side.
- `for i in range(len(y))` — one iteration per row.
- `row = A[i]` — the current row as `[size, 1]`.
- `AtA = AtA + np.outer(row, row)` — every pair of columns multiplied together.
- `Aty = Aty + row * y[i]` — each column multiplied by this row's target.
- `print(f"row {i} ...")` — the running sums after each row.
- `w_hand = np.linalg.solve(...)` — two equations, two unknowns.
- `print("by hand" ...)` — slope and intercept from the accumulated sums.
- `print("lstsq" ...)` — the same numbers, reached differently.

## 3. Plot: the logistic S-curve and its boundary

For a yes/no target the line's value is squashed by `1 / (1 + exp(-z))`, which is always between 0 and 1. Fitting the two numbers by gradient descent gives a curve that crosses 0.5 once; that crossing is the decision boundary, and here it sits where the labels switch.

In [ ]:
xc = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0])   # feature: hours studied
yc = np.array([0, 0, 0, 0, 0, 1, 1, 1, 1, 1], dtype=float)              # label: fail or pass

def sigmoid(z):                                           # the squashing function
    return 1 / (1 + np.exp(-z))                           # any real number in, a probability out

wl, bl = 0.0, 0.0                                         # start the logistic weights at zero
for step in range(3000):                                  # gradient descent on the log loss
    p = sigmoid(wl * xc + bl)                             # predicted probability for every row
    wl = wl - 0.5 * np.mean((p - yc) * xc)                # nudge the weight downhill
    bl = bl - 0.5 * np.mean(p - yc)                       # nudge the bias downhill
print("logistic slope, bias:", round(wl, 3), round(bl, 3))   # steepness and offset
print("boundary at x =", round(-bl / wl, 3))              # the input where p equals 0.5
grid = np.linspace(0, 6, 200)                             # a dense grid for drawing
import matplotlib.pyplot as plt                           # the plotting library
plt.figure(figsize=(6, 3))                                # a wide, short figure
plt.plot(grid, sigmoid(wl * grid + bl), color="tab:red", label="predicted probability")   # the S-curve
plt.scatter(xc, yc, color="tab:blue", label="data (0 or 1)")   # the ten labelled rows
plt.axhline(0.5, color="grey", linestyle="--")            # the 50 percent line
plt.axvline(-bl / wl, color="grey", linestyle=":")        # the boundary it defines
plt.xlabel("hours studied")                               # x-axis label
plt.ylabel("probability of passing")                      # y-axis label
plt.legend(fontsize=8)                                    # name the curve and the points
plt.title("One crossing at 0.5 makes the yes/no decision")   # the takeaway
plt.show()                                                # render

**Line by line**

- `xc = ...` — the feature for the classification problem.
- `yc = ...` — the label, coded 0 for fail and 1 for pass.
- `def sigmoid(z)` — names the squashing function.
- `return 1 / (1 + np.exp(-z))` — the logistic curve, bounded between 0 and 1.
- `wl, bl = 0.0, 0.0` — starting values for the two fitted numbers.
- `for step in range(3000)` — plain gradient descent, no library solver.
- `p = sigmoid(...)` — current probabilities.
- `wl = wl - ...` — the weight moves against the gradient of the log loss.
- `bl = bl - ...` — the bias does the same.
- `print("logistic slope, bias" ...)` — the fitted pair.
- `print("boundary at x" ...)` — solve `wl*x + bl = 0` for the 50 percent point.
- `grid = np.linspace(0, 6, 200)` — a smooth range for the curve.
- `import matplotlib.pyplot as plt` — the plotting library.
- `plt.figure(...)` — start a figure.
- `plt.plot(...)` — the S-curve across that range.
- `plt.scatter(...)` — the data, sitting at exactly 0 or 1.
- `plt.axhline(0.5, ...)` — the 50 percent decision line.
- `plt.axvline(...)` — the boundary it implies.
- `plt.xlabel(...)` — the input.
- `plt.ylabel(...)` — the fitted probability.
- `plt.legend(...)` — names the curve and the points.
- `plt.title(...)` — states what the crossing means.
- `plt.show()` — renders the figure.

## 4. Interactive: move the threshold

0.5 is a convention, not a law. The slider changes the probability you require before saying yes; the green line slides along the curve and the points recolour as their predictions flip. The title reports the accuracy you get at that threshold.

In [ ]:
from ipywidgets import interact                            # sliders that rerun a function
grid = np.linspace(0, 6, 200)                             # the same dense drawing grid
p_all = sigmoid(wl * xc + bl)                             # reuse the fit from section 3

def show(thr=0.5):                                        # thr comes from the slider
    pred = (p_all >= thr).astype(int)                     # 1 where the probability clears the bar
    acc = np.mean(pred == yc)                             # share of rows called correctly
    cut = (np.log(thr / (1 - thr)) - bl) / wl             # invert the sigmoid to find the x value
    plt.figure(figsize=(6, 3))                            # a fresh figure for each drag
    plt.plot(grid, sigmoid(wl * grid + bl), color="tab:red")   # the fitted S-curve
    plt.axhline(thr, color="tab:orange", linestyle="--")  # your chosen probability
    plt.axvline(cut, color="tab:green", linestyle=":")    # where the curve crosses it
    plt.scatter(xc, yc, c=pred, cmap="coolwarm", edgecolor="k", s=60)   # colour = prediction
    plt.xlabel("hours studied")                           # x-axis label
    plt.ylabel("probability of passing")                  # y-axis label
    plt.title(f"threshold {thr:.2f}: boundary at {cut:.2f}, accuracy {acc:.0%}")   # live readout
    plt.show()                                            # draw the frame

interact(show, thr=(0.1, 0.9, 0.05))                      # attach the slider

**Line by line**

- `from ipywidgets import interact` — the widget helper.
- `grid = np.linspace(0, 6, 200)` — the drawing grid.
- `p_all = sigmoid(...)` — the ten fitted probabilities, computed once.
- `def show(thr=0.5)` — redraws one frame per slider value.
- `pred = (...).astype(int)` — hard 0/1 decisions at this threshold.
- `acc = np.mean(...)` — accuracy, as a fraction of ten rows.
- `cut = ...` — inverts the sigmoid: `log(thr/(1-thr))` is the `z` needed, then solve for `x`.
- `plt.figure(...)` — a fresh figure per drag.
- `plt.plot(...)` — the unchanging curve.
- `plt.axhline(thr, ...)` — your threshold as a horizontal line.
- `plt.axvline(cut, ...)` — the boundary it produces.
- `plt.scatter(..., c=pred, cmap=...)` — points coloured by their decision.
- `plt.xlabel(...)` — the input.
- `plt.ylabel(...)` — the fitted probability.
- `plt.title(f"...")` — threshold, boundary and accuracy, updated live.
- `plt.show()` — draws the frame.
- `interact(show, thr=(...))` — slider from 0.1 to 0.9 in steps of 0.05.

## 5. When it breaks: squared error on a 0/1 target

Squared error treats a probability like any other number, so it is happy with predictions above 1 and below 0 — which is what least squares produces on labels. Worse, it charges a confidently wrong probability less than one unit, while log loss grows without limit.

In [ ]:
Ac = np.column_stack([xc, np.ones_like(xc)])              # design matrix for the 0/1 data
line = np.linalg.lstsq(Ac, yc, rcond=None)[0]             # best straight line for the labels
fit = Ac @ line                                           # its outputs are heights, not probabilities
print("fitted line:", np.round(line, 3))                  # slope and intercept
print("smallest prediction:", round(fit.min(), 3))        # below zero: impossible
print("largest  prediction:", round(fit.max(), 3))        # above one: impossible
print("MSE  for p=0.40 when y=1:", round((0.40 - 1) ** 2, 3))    # a mild miss
print("MSE  for p=0.02 when y=1:", round((0.02 - 1) ** 2, 3))    # a confident miss, barely worse
print("log loss for p=0.40:", round(-np.log(0.40), 3))    # mild miss
print("log loss for p=0.02:", round(-np.log(0.02), 3))    # punished far harder
plt.figure(figsize=(6, 3))                                # a wide, short figure
plt.scatter(xc, yc, color="tab:blue", label="labels")     # the 0/1 truth
plt.plot(xc, fit, color="tab:red", label="least squares line")   # the line that ignores the bounds
plt.axhspan(0, 1, color="tab:green", alpha=0.12, label="valid probability band")   # 0 to 1
plt.xlabel("hours studied")                               # x-axis label
plt.ylabel("prediction")                                  # y-axis label
plt.legend(fontsize=8, loc="upper left")                  # name the elements
plt.title("A line on 0/1 labels predicts probabilities below 0 and above 1")   # the takeaway
plt.show()                                                # render

**Line by line**

- `Ac = np.column_stack(...)` — same design matrix shape as section 1.
- `line = np.linalg.lstsq(...)` — least squares applied to yes/no labels.
- `fit = Ac @ line` — the fitted values themselves.
- `print("fitted line" ...)` — slope and intercept of that line.
- `print("smallest prediction" ...)` — the lowest fitted value, negative.
- `print("largest prediction" ...)` — the highest, above 1.
- `print("MSE for p=0.40" ...)` — squared error on a mild miss.
- `print("MSE for p=0.02" ...)` — squared error on a confident miss: still under 1.
- `print("log loss for p=0.40" ...)` — the same mild miss under log loss.
- `print("log loss for p=0.02" ...)` — the confident miss, punished far harder.
- `plt.figure(...)` — a wide, short figure.
- `plt.scatter(...)` — the labels at 0 and 1.
- `plt.plot(xc, fit, ...)` — the straight line, visibly leaving the legal band.
- `plt.axhspan(0, 1, ...)` — shades the only values a probability may take.
- `plt.xlabel(...)` — the input.
- `plt.ylabel(...)` — the fitted height, not a probability.
- `plt.legend(...)` — names the points, the line and the band.
- `plt.title(...)` — names the failure.
- `plt.show()` — renders the figure.

## Your turn

Set `yc[5]` to 0 in section 3 and refit: watch the boundary move and the accuracy in section 4 peak at a threshold other than 0.5. Then compute the log loss of the least squares line in section 5 after clipping its predictions to `[0.001, 0.999]` — why is that clipping a patch, not a fix?